In [1]:
import pandas as pd

occupations = pd.DataFrame({
    "occ_code": ["15-1252", "29-1141", "41-2011", "19-3011", "53-3032"],
    "occupation": ["Software Developers", "Registered Nurses", "Cashiers", "Economists", "Truck Drivers"],
    "median_wage": [132270, 86070, 29720, 115730, 54320]
})

ai_usage = pd.DataFrame({
    "occ_code": ["15-1252", "29-1141", "19-3011", "25-2021"],
    "usage_share": [0.182, 0.031, 0.094, 0.055]
})

In [2]:
occupations

,occ_code,occupation,median_wage
0,15-1252,Software Developers,132270
1,29-1141,Registered Nurses,86070
2,41-2011,Cashiers,29720
3,19-3011,Economists,115730
4,53-3032,Truck Drivers,54320


In [3]:
ai_usage

,occ_code,usage_share
0,15-1252,0.182
1,29-1141,0.031
2,19-3011,0.094
3,25-2021,0.055


In [4]:
# Prediction: 9 rows, all occupations survive
merged = pd.merge(occupations, ai_usage, on="occ_code")
merged

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270,0.182
1,29-1141,Registered Nurses,86070,0.031
2,19-3011,Economists,115730,0.094


In [ ]:
# Prediction: 5 rows, all of left table
pd.merge(occupations, ai_usage, on="occ_code", how="left")

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270,0.182
1,29-1141,Registered Nurses,86070,0.031
2,41-2011,Cashiers,29720,NaN
3,19-3011,Economists,115730,0.094
4,53-3032,Truck Drivers,54320,NaN


In [ ]:
# Prediction: 3 rows, matches only
pd.merge(occupations, ai_usage, on="occ_code", how="inner")

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270,0.182
1,29-1141,Registered Nurses,86070,0.031
2,19-3011,Economists,115730,0.094


In [ ]:
# Prediction: 4 rows, all of right table
pd.merge(occupations, ai_usage, on="occ_code", how="right")

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270.0,0.182
1,29-1141,Registered Nurses,86070.0,0.031
2,19-3011,Economists,115730.0,0.094
3,25-2021,NaN,NaN,0.055


In [8]:
# Prediction: 6 rows, all of left table and right table
pd.merge(occupations, ai_usage, on="occ_code", how="outer")

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270.0,0.182
1,19-3011,Economists,115730.0,0.094
2,25-2021,NaN,NaN,0.055
3,29-1141,Registered Nurses,86070.0,0.031
4,41-2011,Cashiers,29720.0,NaN
5,53-3032,Truck Drivers,54320.0,NaN


Question: for analyzing wages and AI use, which merge option would you want and why?
My answer: inner because you cannot do any analysis on the relationship between wages and AI usage if the data point for one is missing

In [9]:
print("left:", len(occupations), "right:", len(ai_usage), "merged:", len(merged))

left: 5 right: 4 merged: 3


In [10]:
ai_usage_dup = pd.concat([ai_usage, ai_usage.head(1)])

In [11]:
# Prediction: 3 rows, matches only, but one duplicate row
pd.merge(occupations, ai_usage_dup, on="occ_code", how="inner")

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270,0.182
1,15-1252,Software Developers,132270,0.182
2,29-1141,Registered Nurses,86070,0.031
3,19-3011,Economists,115730,0.094


In [12]:
ai_usage_dup.value_counts("occ_code")

occ_code
15-1252    2
29-1141    1
19-3011    1
25-2021    1
Name: count, dtype: int64

In [15]:
# Objective: produce a table of occupation name, wage, and usage share, for matched occupations only, sorted by usage share descending
pd.merge(occupations, ai_usage, on="occ_code", how="inner").sort_values("usage_share", ascending=False)

,occ_code,occupation,median_wage,usage_share
0,15-1252,Software Developers,132270,0.182
2,19-3011,Economists,115730,0.094
1,29-1141,Registered Nurses,86070,0.031


Question: Which occupation is the AI-usage outlier relative to its wage?
Answer: Among the three matched occupations, wage and usage move together with no outlier standing out. The more notable feature is the non-matches: the two lowest-wage occupations (Cashiers, Truck Drivers) have no usage data at all, and Teachers have usage but no wage row.

Notes: 
- Merging combines data from two or more tables into one table, adding columns when the selected criteria from a row matches
- `left` keeps all rows from the left table, `right` keeps all rows from the right table, `inner` keeps only the matches from both tables, and `outer` keeps all of the rows and adds NaN in missing data points
- Before merging, it's good to check two things: whether or not the key values are unique (`value_counts()`) and whether key values align in the way you would expect (e.g., eyeball whether 13 in one is 13.00 in another)